# Tech Challenge — Frente 5: Inteligência Artificial e oportunidades estratégicas

Pergunta central da narrativa:

**A Inteligência Artificial já faz parte do cotidiano dos profissionais, mas as empresas estão acompanhando essa transformação no mesmo ritmo?**

Este notebook lê as tabelas da camada Gold pelo Glue Data Catalog, realiza as
análises com PySpark e usa Pandas apenas para renderizar tabelas pequenas e gráficos.



In [5]:
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 15

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 15 minutes.


In [1]:
# 1. Inicialização da sessão Spark e bibliotecas
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from awsglue.context import GlueContext

from io import BytesIO
import re
import textwrap
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

print("Spark:", spark.version)



Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 15
Session ID: 50e5dfc1-f2a9-4f4f-9909-245dae94ef46
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 50e5dfc1-f2a9-4f4f-9909-245dae94ef46 to get into ready status...
Session 50e5dfc1-f2a9-4f4f-9909-245dae94ef46 has been created.
Spark: 3.5.4-amzn-0


In [3]:
# 2. Parâmetros do projeto
DB_GOLD = "dados_gold"
AMOSTRA_MINIMA = 30

# Os gráficos aparecem no notebook mesmo quando esta opção é False.
SALVAR_GRAFICOS_S3 = True
BUCKET_GRAFICOS = "tech-challenge-014478672967"
PREFIXO_GRAFICOS = "artefatos/frente_5_ia/graficos"

s3 = boto3.client("s3")


def nome_seguro(texto):
    """Transforma um título em nome simples para arquivo."""
    texto = texto.lower().strip()
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")


def concluir_grafico(fig, nome_arquivo):
    """Exibe o gráfico e, opcionalmente, salva uma cópia PNG no S3."""
    fig.tight_layout()

    if SALVAR_GRAFICOS_S3:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
        buffer.seek(0)
        chave = f"{PREFIXO_GRAFICOS}/{nome_arquivo}.png"

        try:
            s3.put_object(
                Bucket=BUCKET_GRAFICOS,
                Key=chave,
                Body=buffer.getvalue(),
                ContentType="image/png"
            )
            print(f"Gráfico salvo em s3://{BUCKET_GRAFICOS}/{chave}")
        except Exception as erro:
            print("O gráfico foi exibido, mas não foi salvo no S3:", erro)

    plt.show()
    plt.close(fig)


def adicionar_rotulos_barras(ax, casas=1, sufixo="%"):
    """Adiciona rótulos nas barras de gráficos do Matplotlib/Seaborn."""
    for container in ax.containers:
        labels = []
        for barra in container:
            valor = barra.get_height()
            if pd.isna(valor):
                labels.append("")
            else:
                labels.append(f"{valor:.{casas}f}{sufixo}")
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)



## 1. Leitura e validação das tabelas Gold



In [4]:
tabelas = {
    "fato": "ft_ia_profissional",
    "adocao_anual": "agg_ia_adocao_anual",
    "segmentos": "agg_ia_adocao_segmento",
    "maturidade": "agg_ia_maturidade_organizacional",
    "salarios": "agg_ia_salario_senioridade",
    "tecnologias": "agg_ia_tecnologias_senioridade"
}

dfs = {
    apelido: glueContext.create_dynamic_frame.from_catalog(
        database=DB_GOLD,
        table_name=tabela
    ).toDF()
    for apelido, tabela in tabelas.items()
}

for apelido, df in dfs.items():
    print(f"{apelido:15s} | linhas: {df.count():6d} | colunas: {len(df.columns):3d}")

df_fato = dfs["fato"]
df_adocao = dfs["adocao_anual"]
df_segmentos = dfs["segmentos"]
df_maturidade = dfs["maturidade"]
df_salarios = dfs["salarios"]
df_tecnologias = dfs["tecnologias"]



fato            | linhas:  14002 | colunas:  73
adocao_anual    | linhas:      3 | colunas:  19
segmentos       | linhas:     83 | colunas:  13
maturidade      | linhas:     83 | colunas:   9
salarios        | linhas:     20 | colunas:  12
tecnologias     | linhas:     20 | colunas:  12
/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
# Cobertura das perguntas de IA por ano.
# Nulo não é tratado como "não" porque algumas perguntas podem não existir em todos os anos.
df_cobertura = (
    df_fato
    .groupBy("ano_pesquisa")
    .agg(
        F.count("*").alias("total_registros"),
        F.count("flag_usa_ia").alias("respostas_flag_usa_ia"),
        F.count("ia_generativa_prioridade").alias("respostas_prioridade"),
        F.count("ia_bons_resultados_llm").alias("respostas_bons_resultados"),
        F.count("tipo_uso_ia_empresa").alias("respostas_tipo_uso_empresa"),
        F.count("usa_llm_trabalho").alias("respostas_uso_llm_trabalho")
    )
    .withColumn(
        "pct_cobertura_uso_ia",
        F.round(100 * F.col("respostas_flag_usa_ia") / F.col("total_registros"), 2)
    )
    .orderBy("ano_pesquisa")
)

df_cobertura.show(truncate=False)



+------------+---------------+---------------------+--------------------+-------------------------+--------------------------+--------------------------+--------------------+
|ano_pesquisa|total_registros|respostas_flag_usa_ia|respostas_prioridade|respostas_bons_resultados|respostas_tipo_uso_empresa|respostas_uso_llm_trabalho|pct_cobertura_uso_ia|
+------------+---------------+---------------------+--------------------+-------------------------+--------------------------+--------------------------+--------------------+
|2023        |5293           |3772                 |896                 |0                        |3772                      |3772                      |71.26               |
|2024        |5215           |3617                 |1045                |0                        |3617                      |3617                      |69.36               |
|2025        |3494           |2105                 |652                 |646                      |2105                      

## 2. Adoção individual e evolução anual

Responde:

- Qual é o índice de adoção de IA?
- Como a adoção evoluiu por ano?
- Como os profissionais acessam as ferramentas?



In [6]:
janela_ano = Window.orderBy("ano_pesquisa")

df_evolucao = (
    df_adocao
    .withColumn(
        "variacao_adocao_pp",
        F.round(
            F.col("pct_adocao_ia")
            - F.lag("pct_adocao_ia").over(janela_ano),
            2
        )
    )
    .withColumn(
        "variacao_apoio_pp",
        F.round(
            F.col("pct_apoio_institucional_total")
            - F.lag("pct_apoio_institucional_total").over(janela_ano),
            2
        )
    )
    .orderBy("ano_pesquisa")
)

df_evolucao.select(
    "ano_pesquisa",
    "total_registros",
    "respostas_validas_uso_ia",
    "pct_adocao_ia",
    "variacao_adocao_pp",
    "pct_apoio_institucional_total",
    "variacao_apoio_pp",
    "gap_adocao_apoio_pp"
).show(truncate=False)



+------------+---------------+------------------------+-------------+------------------+-----------------------------+-----------------+-------------------+
|ano_pesquisa|total_registros|respostas_validas_uso_ia|pct_adocao_ia|variacao_adocao_pp|pct_apoio_institucional_total|variacao_apoio_pp|gap_adocao_apoio_pp|
+------------+---------------+------------------------+-------------+------------------+-----------------------------+-----------------+-------------------+
|2023        |5293           |3772                    |80.97        |NULL              |6.36                         |NULL             |74.6               |
|2024        |5215           |3617                    |93.83        |12.86             |19.27                        |12.91            |74.56              |
|2025        |3494           |2105                    |98.15        |4.32              |42.33                        |23.06            |55.82              |
+------------+---------------+------------------------+---

In [7]:
# Gráfico 1 — evolução da adoção e do apoio institucional.
pdf_evolucao = df_evolucao.orderBy("ano_pesquisa").toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    pdf_evolucao["ano_pesquisa"],
    pdf_evolucao["pct_adocao_ia"],
    marker="o",
    linewidth=2.5,
    label="Profissionais que usam IA"
)
ax.plot(
    pdf_evolucao["ano_pesquisa"],
    pdf_evolucao["pct_apoio_institucional_total"],
    marker="o",
    linewidth=2.5,
    label="Uso com ferramenta paga pela empresa"
)

for _, linha in pdf_evolucao.iterrows():
    ax.annotate(
        f"{linha['pct_adocao_ia']:.1f}%",
        (linha["ano_pesquisa"], linha["pct_adocao_ia"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

ax.set_title("Adoção individual de IA versus apoio institucional")
ax.set_xlabel("Ano da pesquisa")
ax.set_ylabel("Percentual (%)")
ax.set_ylim(0, 100)
ax.legend()
concluir_grafico(fig, "01_adocao_versus_apoio_institucional")



Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/01_adocao_versus_apoio_institucional.png


In [8]:
# Gráfico 2 — modalidades de acesso entre usuários de IA.
colunas_acesso = {
    "pct_copilot_entre_usuarios_ia": "Copilot",
    "pct_gratuitas_entre_usuarios_ia": "Ferramentas gratuitas",
    "pct_paga_empresa_entre_usuarios_ia": "Paga pela empresa",
    "pct_paga_proprio_entre_usuarios_ia": "Paga pelo profissional"
}

pdf_acesso = pdf_evolucao[["ano_pesquisa"] + list(colunas_acesso)].rename(
    columns=colunas_acesso
)
pdf_acesso = pdf_acesso.melt(
    id_vars="ano_pesquisa",
    var_name="Modalidade",
    value_name="Percentual"
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=pdf_acesso,
    x="ano_pesquisa",
    y="Percentual",
    hue="Modalidade",
    ax=ax
)
ax.set_title("Modalidades de acesso às ferramentas de IA")
ax.set_xlabel("Ano da pesquisa")
ax.set_ylabel("Percentual entre usuários de IA (%)")
ax.set_ylim(0, 110)
adicionar_rotulos_barras(ax)
ax.legend(title="Modalidade", bbox_to_anchor=(1.02, 1), loc="upper left")
concluir_grafico(fig, "02_modalidades_acesso_ia")



Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/02_modalidades_acesso_ia.png


In [9]:

linhas_anuais = df_evolucao.orderBy("ano_pesquisa").collect()

if linhas_anuais:
    primeiro = linhas_anuais[0]
    ultimo = linhas_anuais[-1]
    variacao_periodo = round(ultimo["pct_adocao_ia"] - primeiro["pct_adocao_ia"], 2)

    print("NARRATIVA — ADOÇÃO")
    print(
        f"No ano mais recente ({ultimo['ano_pesquisa']}), "
        f"{ultimo['pct_adocao_ia']:.2f}% dos respondentes com resposta válida "
        "declararam utilizar IA."
    )
    print(
        f"Em relação a {primeiro['ano_pesquisa']}, a variação acumulada foi de "
        f"{variacao_periodo:+.2f} pontos percentuais."
    )
    print(
        f"O apoio institucional alcançou {ultimo['pct_apoio_institucional_total']:.2f}% "
        f"da amostra válida, deixando um gap de {ultimo['gap_adocao_apoio_pp']:.2f} "
        "pontos percentuais entre adoção e financiamento empresarial."
    )



NARRATIVA — ADOÇÃO
No ano mais recente (2025), 98.15% dos respondentes com resposta válida declararam utilizar IA.
Em relação a 2023, a variação acumulada foi de +17.18 pontos percentuais.
O apoio institucional alcançou 42.33% da amostra válida, deixando um gap de 55.82 pontos percentuais entre adoção e financiamento empresarial.


## 3. Diferenças por senioridade, região, modelo de trabalho e cargo

Apenas grupos com pelo menos `AMOSTRA_MINIMA` respostas válidas são usados na
comparação principal. Isso evita destacar percentuais extremos de grupos muito pequenos.



In [10]:
df_segmentos_validos = (
    df_segmentos
    .filter(F.col("respostas_validas_uso_ia") >= AMOSTRA_MINIMA)
    .orderBy("ano_pesquisa", "tipo_dimensao", F.desc("pct_adocao_ia"))
)

df_segmentos_validos.show(100, truncate=False)



+------------+------------------+--------------------------------------------------------------------------------------------------------------+---------------+------------------------+-----------+--------------------------+------------------------------------+-----------------------------+-------------+--------------------------------+----------------------------------------------+-------------------+
|ano_pesquisa|tipo_dimensao     |categoria                                                                                                     |total_registros|respostas_validas_uso_ia|usuarios_ia|usuarios_com_apoio_empresa|usuarios_autofinanciamento_exclusivo|pct_cobertura_pergunta_uso_ia|pct_adocao_ia|pct_apoio_empresa_entre_usuarios|pct_autofinanciamento_exclusivo_entre_usuarios|gap_adocao_apoio_pp|
+------------+------------------+--------------------------------------------------------------------------------------------------------------+---------------+------------------------+---

In [11]:
def analisar_segmento(tipo_dimensao, max_categorias=12):
    """Mostra e plota as categorias do ano mais recente de uma dimensão."""
    df_dimensao = df_segmentos_validos.filter(
        F.col("tipo_dimensao") == tipo_dimensao
    )

    ano_mais_recente = df_dimensao.agg(F.max("ano_pesquisa")).first()[0]

    if ano_mais_recente is None:
        print(f"Sem dados válidos para {tipo_dimensao}.")
        return None

    df_ano = (
        df_dimensao
        .filter(F.col("ano_pesquisa") == ano_mais_recente)
        .orderBy(F.desc("respostas_validas_uso_ia"))
        .limit(max_categorias)
    )

    pdf = df_ano.toPandas().sort_values("pct_adocao_ia", ascending=True)

    if pdf.empty:
        print(f"Sem dados válidos para {tipo_dimensao}.")
        return None

    fig, ax = plt.subplots(figsize=(10, max(5, len(pdf) * 0.48)))
    sns.barplot(
        data=pdf,
        x="pct_adocao_ia",
        y="categoria",
        color="#3B82F6",
        ax=ax
    )
    ax.set_title(f"Adoção de IA por {tipo_dimensao.lower()} — {ano_mais_recente}")
    ax.set_xlabel("Adoção de IA (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)

    concluir_grafico(
        fig,
        f"03_adocao_{nome_seguro(tipo_dimensao)}_{ano_mais_recente}"
    )

    maior = pdf.loc[pdf["pct_adocao_ia"].idxmax()]
    menor = pdf.loc[pdf["pct_adocao_ia"].idxmin()]
    diferenca = maior["pct_adocao_ia"] - menor["pct_adocao_ia"]

    print(f"NARRATIVA — {tipo_dimensao.upper()}")
    print(
        f"Em {ano_mais_recente}, a maior adoção ocorreu em "
        f"'{maior['categoria']}' ({maior['pct_adocao_ia']:.2f}%) e a menor em "
        f"'{menor['categoria']}' ({menor['pct_adocao_ia']:.2f}%), uma diferença "
        f"de {diferenca:.2f} pontos percentuais."
    )

    return pdf



In [12]:
pdf_senioridade = analisar_segmento("Senioridade")
pdf_regiao = analisar_segmento("Regiao")
pdf_modelo = analisar_segmento("Modelo de trabalho")
pdf_cargo = analisar_segmento("Cargo", max_categorias=10)



Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/03_adocao_senioridade_2025.png
NARRATIVA — SENIORIDADE
Em 2025, a maior adoção ocorreu em 'Sênior' (98.48%) e a menor em 'Júnior' (97.18%), uma diferença de 1.30 pontos percentuais.
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/03_adocao_regiao_2025.png
NARRATIVA — REGIAO
Em 2025, a maior adoção ocorreu em 'Centro-oeste' (99.31%) e a menor em 'Nordeste' (95.56%), uma diferença de 3.75 pontos percentuais.
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/03_adocao_modelo_de_trabalho_2025.png
NARRATIVA — MODELO DE TRABALHO
Em 2025, a maior adoção ocorreu em 'Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)' (98.70%) e a menor em 'Modelo híbrido com dias fixos de trabalho presencial' (97.63%), uma diferença de 1.07 pontos percentuais.
Gráfico salvo em s3://tech-challenge-014478672967

## 4. Uso organizacional, prioridade, resultados e maturidade

Responde:

- A empresa financia ferramentas?
- O uso é individual ou institucional?
- IA é prioridade estratégica?
- Há bons resultados com LLMs?
- Como a IA é utilizada dentro das empresas?

Prioridade e resultados são distribuições exclusivas. As formas de uso são
multisseleção e, por isso, seus percentuais podem somar mais de 100%.



In [13]:
df_maturidade.orderBy(
    "ano_pesquisa",
    "indicador",
    F.desc("quantidade_resposta")
).show(200, truncate=False)



+------------+-------------------------------+----------------------+--------------------------------------------+-------------------+---------------------------------+-------------------+---------------------------+-----------------------+
|ano_pesquisa|indicador                      |tipo_metrica          |resposta                                    |quantidade_resposta|total_respostas_validas_indicador|total_registros_ano|pct_entre_respostas_validas|pct_cobertura_indicador|
+------------+-------------------------------+----------------------+--------------------------------------------+-------------------+---------------------------------+-------------------+---------------------------+-----------------------+
|2023        |Formas de uso de IA na empresa |Multisselecao         |Uso individual sem centralizacao            |1723               |3772                             |5293               |45.68                      |71.26                  |
|2023        |Formas de uso de IA na

In [14]:
def grafico_maturidade(indicador):
    """Plota a distribuição das respostas no ano mais recente disponível."""
    df_indicador = df_maturidade.filter(F.col("indicador") == indicador)
    ano_mais_recente = df_indicador.agg(F.max("ano_pesquisa")).first()[0]

    if ano_mais_recente is None:
        print(f"Sem dados para: {indicador}")
        return None

    pdf = (
        df_indicador
        .filter(F.col("ano_pesquisa") == ano_mais_recente)
        .orderBy(F.desc("pct_entre_respostas_validas"))
        .toPandas()
    )

    if pdf.empty:
        return None

    pdf["resposta_grafico"] = pdf["resposta"].astype(str).apply(
        lambda valor: textwrap.fill(valor, width=42)
    )
    pdf = pdf.sort_values("pct_entre_respostas_validas", ascending=True)

    fig, ax = plt.subplots(figsize=(11, max(5, len(pdf) * 0.6)))
    sns.barplot(
        data=pdf,
        x="pct_entre_respostas_validas",
        y="resposta_grafico",
        color="#6366F1",
        ax=ax
    )
    ax.set_title(f"{indicador} — {ano_mais_recente}")
    ax.set_xlabel("Percentual entre respostas válidas (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)

    cobertura = pdf["pct_cobertura_indicador"].iloc[0]
    ax.text(
        0.99,
        -0.12,
        f"Cobertura da pergunta: {cobertura:.1f}%",
        transform=ax.transAxes,
        ha="right",
        fontsize=9
    )

    concluir_grafico(
        fig,
        f"04_{nome_seguro(indicador)}_{ano_mais_recente}"
    )
    return pdf



In [15]:
indicadores_maturidade = [
    "Perfil de acesso a IA",
    "Prioridade de IA generativa",
    "Bons resultados com LLM",
    "Maturidade organizacional de IA",
    "Formas de uso de IA na empresa"
]

resultados_maturidade = {
    indicador: grafico_maturidade(indicador)
    for indicador in indicadores_maturidade
}



Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/04_perfil_de_acesso_a_ia_2025.png
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/04_prioridade_de_ia_generativa_2025.png
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/04_bons_resultados_com_llm_2025.png
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/04_maturidade_organizacional_de_ia_2025.png
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/04_formas_de_uso_de_ia_na_empresa_2025.png


In [16]:
# Texto automático para a relação entre uso individual e uso organizacional.
df_formas_uso = df_maturidade.filter(
    F.col("indicador") == "Formas de uso de IA na empresa"
)
ano_formas_uso = df_formas_uso.agg(F.max("ano_pesquisa")).first()[0]

if ano_formas_uso is not None:
    linhas_formas = {
        linha["resposta"]: linha["pct_entre_respostas_validas"]
        for linha in (
            df_formas_uso
            .filter(F.col("ano_pesquisa") == ano_formas_uso)
            .collect()
        )
    }

    pct_individual = linhas_formas.get("Uso individual sem centralizacao")
    pct_organizacional = linhas_formas.get("Uso organizacional estruturado")
    pct_centralizado = linhas_formas.get(
        "Direcionamento centralizado e apoio a custos"
    )

    print("NARRATIVA — USO INDIVIDUAL VERSUS ORGANIZACIONAL")
    if pct_individual is not None:
        print(
            f"Em {ano_formas_uso}, {pct_individual:.2f}% das respostas válidas "
            "mencionaram uso individual sem centralização."
        )
    if pct_organizacional is not None:
        print(
            f"No mesmo ano, {pct_organizacional:.2f}% mencionaram pelo menos uma "
            "forma de uso organizacional estruturado."
        )
    if pct_centralizado is not None:
        print(
            f"O direcionamento centralizado com apoio aos custos apareceu em "
            f"{pct_centralizado:.2f}% das respostas válidas."
        )
    print(
        "Como a pergunta aceita múltiplas respostas, uso individual e organizacional "
        "podem aparecer simultaneamente e os percentuais não devem ser somados."
    )



NARRATIVA — USO INDIVIDUAL VERSUS ORGANIZACIONAL
Em 2025, 48.31% das respostas válidas mencionaram uso individual sem centralização.
No mesmo ano, 67.79% mencionaram pelo menos uma forma de uso organizacional estruturado.
O direcionamento centralizado com apoio aos custos apareceu em 35.68% das respostas válidas.
Como a pergunta aceita múltiplas respostas, uso individual e organizacional podem aparecer simultaneamente e os percentuais não devem ser somados.


## 5. Associação entre IA, salário e senioridade

A pesquisa é observacional. Portanto, esta etapa identifica associação, não causalidade.
A mediana é priorizada porque salários são normalmente assimétricos e têm outliers.



In [24]:
df_salarios_validos = df_salarios.filter(
    F.col("quantidade_profissionais") >= AMOSTRA_MINIMA
)

pares_salario = (
    df_salarios_validos
    .groupBy(
        "ano_pesquisa",
        "nivel_comparavel",
        "nivel_ordem"
    )
    .agg(
        F.countDistinct("flag_usa_ia").alias("quantidade_grupos")
    )
    .filter(F.col("quantidade_grupos") == 2)
)

anos_salario_completos = (
    pares_salario
    .groupBy("ano_pesquisa")
    .agg(
        F.count("*").alias("senioridades_comparaveis")
    )
    .filter(F.col("senioridades_comparaveis") >= 3)
)

ano_salario = (
    anos_salario_completos
    .agg(F.max("ano_pesquisa").alias("ano"))
    .first()["ano"]
)

senioridades_salario = (
    pares_salario
    .filter(F.col("ano_pesquisa") == ano_salario)
    .select("nivel_comparavel", "nivel_ordem")
)

df_salario_comparavel = (
    df_salarios_validos
    .filter(F.col("ano_pesquisa") == ano_salario)
    .join(
        senioridades_salario,
        ["nivel_comparavel", "nivel_ordem"],
        "inner"
    )
    .orderBy("nivel_ordem", "flag_usa_ia")
)

pdf_salarios = df_salario_comparavel.toPandas()

print(f"Ano selecionado para comparação salarial: {ano_salario}")

df_salario_comparavel.show(100, truncate=False)

Ano selecionado para comparação salarial: 2024
+----------------+-----------+------------+-----------+------------+------------------------+-------------------+------------------------+---------------+------------------------+------------------------+------------------------+
|nivel_comparavel|nivel_ordem|ano_pesquisa|flag_usa_ia|grupo_uso_ia|quantidade_profissionais|salario_medio_grupo|salario_primeiro_quartil|salario_mediano|salario_terceiro_quartil|salario_minimo_observado|salario_maximo_observado|
+----------------+-----------+------------+-----------+------------+------------------------+-------------------+------------------------+---------------+------------------------+------------------------+------------------------+
|Júnior          |1          |2024        |0          |Nao usa IA  |41                      |4268.77            |2500.5                  |4438.0         |5000.5                  |500.0                   |14000.5                 |
|Júnior          |1          |202

In [25]:
if not pdf_salarios.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(
        data=pdf_salarios,
        x="nivel_comparavel",
        y="salario_mediano",
        hue="grupo_uso_ia",
        ax=ax
    )
    ax.set_title(f"Mediana salarial por senioridade e uso de IA — {ano_salario}")
    ax.set_xlabel("Senioridade comparável")
    ax.set_ylabel("Salário mediano")
    ax.tick_params(axis="x", rotation=20)
    adicionar_rotulos_barras(ax, casas=0, sufixo="")
    ax.legend(title="Grupo")
    concluir_grafico(fig, f"05_salario_senioridade_ia_{ano_salario}")

    comparacao_salario = pdf_salarios.pivot_table(
        index=["nivel_comparavel", "nivel_ordem"],
        columns="grupo_uso_ia",
        values="salario_mediano"
    ).reset_index()

    if {"Usa IA", "Nao usa IA"}.issubset(comparacao_salario.columns):
        comparacao_salario["diferenca_percentual"] = (
            100
            * (comparacao_salario["Usa IA"] - comparacao_salario["Nao usa IA"])
            / comparacao_salario["Nao usa IA"]
        ).round(2)

        print("Comparação salarial dentro da mesma senioridade:")
        print(comparacao_salario.to_string(index=False))
        print(
            "Interpretação: diferenças salariais são associações. "
            "Não conclua que o uso de IA causou o salário observado."
        )



<Axes: xlabel='nivel_comparavel', ylabel='salario_mediano'>
Text(0.5, 1.0, 'Mediana salarial por senioridade e uso de IA — 2024')
Text(0.5, 0, 'Senioridade comparável')
Text(0, 0.5, 'Salário mediano')
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/05_salario_senioridade_ia_2024.png
Comparação salarial dentro da mesma senioridade:
nivel_comparavel  nivel_ordem  Nao usa IA   Usa IA  diferenca_percentual
          Júnior            1      4438.0  3584.99                -19.22
           Pleno            2      7000.5  7000.50                  0.00
          Sênior            3     14000.5 14000.50                  0.00
Interpretação: diferenças salariais são associações. Não conclua que o uso de IA causou o salário observado.


## 6. Associação entre IA e amplitude tecnológica

Verifica se usuários de IA também utilizam um conjunto mais amplo de linguagens,
bancos de dados, ferramentas de BI e plataformas de cloud.



In [22]:
df_tecnologias_validas = df_tecnologias.filter(
    F.col("quantidade_profissionais") >= AMOSTRA_MINIMA
)

pares_tecnologia = (
    df_tecnologias_validas
    .groupBy(
        "ano_pesquisa",
        "nivel_comparavel",
        "nivel_ordem"
    )
    .agg(
        F.countDistinct("flag_usa_ia").alias("quantidade_grupos")
    )
    .filter(F.col("quantidade_grupos") == 2)
)

anos_tecnologia_completos = (
    pares_tecnologia
    .groupBy("ano_pesquisa")
    .agg(
        F.count("*").alias("senioridades_comparaveis")
    )
    .filter(F.col("senioridades_comparaveis") >= 3)
)

ano_tecnologia = (
    anos_tecnologia_completos
    .agg(F.max("ano_pesquisa").alias("ano"))
    .first()["ano"]
)

senioridades_tecnologia = (
    pares_tecnologia
    .filter(F.col("ano_pesquisa") == ano_tecnologia)
    .select("nivel_comparavel", "nivel_ordem")
)

df_tecnologias_comparavel = (
    df_tecnologias_validas
    .filter(F.col("ano_pesquisa") == ano_tecnologia)
    .join(
        senioridades_tecnologia,
        ["nivel_comparavel", "nivel_ordem"],
        "inner"
    )
    .orderBy("nivel_ordem", "flag_usa_ia")
)

pdf_tecnologias = df_tecnologias_comparavel.toPandas()

print(f"Ano selecionado para comparação tecnológica: {ano_tecnologia}")

df_tecnologias_comparavel.show(100, truncate=False)

Ano selecionado para comparação tecnológica: 2024
+----------------+-----------+------------+-----------+------------+------------------------+--------------------+-----------------------+-----------------------+---------------------------+-----------------------+-------------------------+
|nivel_comparavel|nivel_ordem|ano_pesquisa|flag_usa_ia|grupo_uso_ia|quantidade_profissionais|media_tecnologias_bi|media_tecnologias_cloud|media_tecnologias_banco|media_tecnologias_linguagem|media_tecnologias_total|mediana_tecnologias_total|
+----------------+-----------+------------+-----------+------------+------------------------+--------------------+-----------------------+-----------------------+---------------------------+-----------------------+-------------------------+
|Júnior          |1          |2024        |0          |Nao usa IA  |41                      |1.24                |1.15                   |2.17                   |1.44                       |6.0                    |5            

In [23]:
if not pdf_tecnologias.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(
        data=pdf_tecnologias,
        x="nivel_comparavel",
        y="media_tecnologias_total",
        hue="grupo_uso_ia",
        ax=ax
    )
    ax.set_title(
        f"Amplitude tecnológica por senioridade e uso de IA — {ano_tecnologia}"
    )
    ax.set_xlabel("Senioridade comparável")
    ax.set_ylabel("Média de tecnologias marcadas")
    ax.tick_params(axis="x", rotation=20)
    adicionar_rotulos_barras(ax, casas=1, sufixo="")
    ax.legend(title="Grupo")
    concluir_grafico(fig, f"06_tecnologias_senioridade_ia_{ano_tecnologia}")

    comparacao_tecnologia = pdf_tecnologias.pivot_table(
        index=["nivel_comparavel", "nivel_ordem"],
        columns="grupo_uso_ia",
        values="media_tecnologias_total"
    ).reset_index()

    if {"Usa IA", "Nao usa IA"}.issubset(comparacao_tecnologia.columns):
        comparacao_tecnologia["diferenca_media_tecnologias"] = (
            comparacao_tecnologia["Usa IA"]
            - comparacao_tecnologia["Nao usa IA"]
        ).round(2)
        print(comparacao_tecnologia.to_string(index=False))



<Axes: xlabel='nivel_comparavel', ylabel='media_tecnologias_total'>
Text(0.5, 1.0, 'Amplitude tecnológica por senioridade e uso de IA — 2024')
Text(0.5, 0, 'Senioridade comparável')
Text(0, 0.5, 'Média de tecnologias marcadas')
Gráfico salvo em s3://tech-challenge-014478672967/artefatos/frente_5_ia/graficos/06_tecnologias_senioridade_ia_2024.png
nivel_comparavel  nivel_ordem  Nao usa IA  Usa IA  diferenca_media_tecnologias
          Júnior            1        6.00    6.54                         0.54
           Pleno            2        6.68    7.81                         1.13
          Sênior            3        8.87    8.71                        -0.16


## 7. Síntese estratégica e narrativa final

Esta seção reúne os principais achados quantitativos. As distribuições de
prioridade, bons resultados e tipo de uso devem ser interpretadas mantendo os
rótulos originais apresentados nos gráficos de maturidade.



In [26]:
print("=" * 80)
print("SÍNTESE DA FRENTE 5 — INTELIGÊNCIA ARTIFICIAL")
print("=" * 80)

if linhas_anuais:
    ultimo = linhas_anuais[-1]
    print(
        f"1. ADOÇÃO: em {ultimo['ano_pesquisa']}, "
        f"{ultimo['pct_adocao_ia']:.2f}% declararam usar IA."
    )
    print(
        f"2. APOIO EMPRESARIAL: {ultimo['pct_apoio_institucional_total']:.2f}% "
        "da amostra válida usava ferramenta paga pela empresa."
    )
    print(
        f"3. GAP: adoção e apoio institucional estavam separados por "
        f"{ultimo['gap_adocao_apoio_pp']:.2f} pontos percentuais."
    )
    print(
        f"4. AUTOFINANCIAMENTO: {ultimo['pct_autofinanciamento_exclusivo_entre_usuarios']:.2f}% "
        "dos usuários pagavam exclusivamente com recursos próprios."
    )

print("\nOPORTUNIDADES E DESAFIOS A CONFRONTAR COM OS RESULTADOS")
print(
    "- Se adoção > apoio empresarial: estruturar licenças corporativas, "
    "governança, segurança e proteção de dados."
)
print(
    "- Se prioridade estratégica > bons resultados: transformar intenção em "
    "casos de uso, treinamento, integração a processos e métricas de retorno."
)
print(
    "- Se houver diferenças relevantes por senioridade, região ou modelo de trabalho: "
    "direcionar capacitação e acesso aos grupos menos atendidos."
)
print(
    "- Se usuários de IA apresentarem maior salário ou amplitude tecnológica: "
    "tratar como associação e oportunidade de capacitação, não como prova causal."
)
print(
    "- Se o autofinanciamento for relevante: investigar uso não padronizado, "
    "custos individuais e risco de exposição de informações corporativas."
)



SÍNTESE DA FRENTE 5 — INTELIGÊNCIA ARTIFICIAL
1. ADOÇÃO: em 2025, 98.15% declararam usar IA.
2. APOIO EMPRESARIAL: 42.33% da amostra válida usava ferramenta paga pela empresa.
3. GAP: adoção e apoio institucional estavam separados por 55.82 pontos percentuais.
4. AUTOFINANCIAMENTO: 22.89% dos usuários pagavam exclusivamente com recursos próprios.

OPORTUNIDADES E DESAFIOS A CONFRONTAR COM OS RESULTADOS
- Se adoção > apoio empresarial: estruturar licenças corporativas, governança, segurança e proteção de dados.
- Se prioridade estratégica > bons resultados: transformar intenção em casos de uso, treinamento, integração a processos e métricas de retorno.
- Se houver diferenças relevantes por senioridade, região ou modelo de trabalho: direcionar capacitação e acesso aos grupos menos atendidos.
- Se usuários de IA apresentarem maior salário ou amplitude tecnológica: tratar como associação e oportunidade de capacitação, não como prova causal.
- Se o autofinanciamento for relevante: investiga